# Data Aggregation and Group Operations
## DAT540 Introduction to Data Science
## University of Stavanger
### L12

In [ ]:
import pandas as pd
import numpy as np

- Categorizing a dataset and applying a function to each group, whether an aggregation or transformation, is a critical component of data analysis workflow
- After loading, merging and preparing a dataset, group stats or pivot tables for reporting or visualization is essential.
- pandas offers a flexible *groupby* interface that enables slicing, dicing and summarizing datasets in a natural way
  - Split a pandas object into pieces using one or more keys
  - Calculate group summary statistics, like count, mean, or standard deviation or a user defined function
  - Apply within-group transformations or other manipulations, like normalization, linear regression, rank, or subset selection
  - Perform quantile analysis and other statistical group analyses

- **GroupBy Mechanics**
- *split-apply-combine* describes group operations
  - split: data in a pandas object (Series, DataFrame) is split into groups based on one or more provided keys
    - The split happens on a particular axis of an object
  - apply: After splitting an object into chunks, a function is applied to each group producing a new value
  - combine: Once a function is applied to the chunks the results are combined into a result object
    - The form of the resulting object will depend on the applied transformation on the data

<img src='images/group_ag.png' width='350'>

- Each grouping can take many forms and the keys do not have to be all of the same type:
  - A list or array of values that is the same length as the axis being grouped
  - A value indicating a column name in a DataFrame
  - a dict or Series giving a correspondence between values on the axis being grouped and the group names
  - A function to be invoked on the axis index or the individual labels in the index
- The *as_index* boolean argument can be set to False to disable the grouping keys represented as index and have them as column values

In [ ]:
df = pd.DataFrame({'key1': ['a', 'a', 'b', 'b', 'a'],
                  'key2': ['one', 'two', 'one', 'two', 'one'],
                  'data1': np.arange(0,5),
                  'data2': np.arange(5,10)})
df
# Let us calculate the mean over key1

In [ ]:
# Using grouping
grouped = df.groupby(df['key1']) # Grouping based on a series
grouped

In [ ]:
grouped[['data1', 'data2']].mean()

In [ ]:
# We can also groupby using multiple keys/series
# Using grouping
grouped = df.groupby([df['key1'], df['key2']])
grouped

In [ ]:
grouped.mean()

In [ ]:
# The size instance method returns a Series containing the size of each group
grouped.size()

In [ ]:
df = pd.DataFrame({'key1': ['a', 'a', 'b', 'b', 'a'],
                  'key2': ['one', 'two', 'one', 'two', 'one'],
                  'data1': np.random.randn(5),
                  'data2': np.random.randn(5)})
df.shape

In [ ]:
df

In [ ]:
# Performing correlation
grouped = df.groupby('key1')
grouped[['data1', 'data2']].corr()

- *Iterating Over Groups*
- The GroupBy object supports iterations, generating a sequence of 2-tuples containing the group name along with the chunk of data

In [ ]:
# Iterating over a df group having a single key
for name, group in df.groupby('key1'):
  print('[name:', name, ']\n')
  print('[group:', group, ']\n\n\n')

In [ ]:
# Iterating over a df group having multiple keys
for (k1, k2), group in df.groupby(['key1', 'key2']):
  print('[k1:', k1, 'k2:', k2, ']\n')
  print('[group:', group, ']\n\n\n')

- Alternatively, a group can be converted into a dict for direct access
- By default axis=0, but it can be changed to any axis

In [ ]:
pieces = dict(list(df.groupby('key1')))
pieces['b']

- *Selecting a Column or Subset of Columns*
- Indexing a GroupBy object created from a DataFrame with column name or array of column names 
  - has the effect of column sub-setting for aggregation
  - especially for large datasets, it may be desirable to aggregate only a few columns
  - the object returned by the indexing operation is a 
    - grouped DataFrame if a list or array is passed 
    - grouped Series if only as single column name is passed as a scalar

In [ ]:
df.groupby('key1')['data1'] # SeriesGroupBy. Alternatively df['data1'].groupby(df['key1'])
df.groupby('key1')[['data1']] # DataFrameGroupBy. Alternatively df[['data1']].groupby(df['key1'])

In [ ]:
# Computing the mean and returning a DataFrame
data = df[['data1']].groupby(df['key1'])
print('type(data1):', type(data.mean()))
# Same as 
data = df.groupby(df['key1'])[['data1']]
print('type(data1):', type(data.mean()))

In [ ]:
# Computing the mean and returing a Series
data = df['data1'].groupby(df['key1'])
type(data.mean())

- *Grouping with Dict and Series*
- Grouping information may exist in a form other than an array

In [ ]:
people = pd.DataFrame(np.random.randn(5, 5,),
                     columns=['a', 'b', 'c', 'd', 'e'],
                     index=['Joe', 'Steve', 'Wes', 'Jim', 'Travis'])
people.iloc[2:3, [1,2]] = np.nan # Add a few NA values
people

In [ ]:
# Suppose we have a seperate dict that maps the columns name of people to another value
mapping = {'a': 'red', 'b': 'red', 'c': 'blue',
          'd': 'blue', 'e':'red', 'f':'orange'}

In [ ]:
# We can instead use this new mapping to group the dataframe people accordingly
by_col = people.groupby(mapping, axis=1)
# In this case we sum columns based on their color representation
by_col.sum()

In [ ]:
# The same functionality also holds true for Series.
# In case of Series the dict keys will be the row indices
map_series = pd.Series(mapping)
map_series

In [ ]:
# When we use the Series to group the DataFrame people, the row indices will be aligned
by_ser = people.T.groupby(map_series)
by_ser.sum() 

- *Grouping with Functions*
- Using python functions is a more generic way of defining group mappings
- Any function passed as a group key will be called once per index with the return value used as group names
- Additionally, functions, dicts, Series all can be combined to create a multi key grouping

In [ ]:
# Grouping people DataFrame by the length of names
people

In [ ]:
# Grouping people DataFrame by the length of names and summing up the corresponding rows
people.groupby(len).sum()

In [ ]:
# Mixing function, list for grouping
key_list = ['one', 'one', 'one', 'two', 'two']
# according to the key list the first three rows will be grouped under 'one' and the last two rows under 'two'
people.groupby([len, key_list]).min()

- *Grouping by Index Levels*
- Aggregating using one of the levels of an axis index for hierarchically indexed datasets

In [ ]:
columns = pd.MultiIndex.from_arrays([['US', 'US', 'US', 'JP', 'JP'],
                                    [1, 3, 5, 1, 3]],
                                    names=['cty', 'tenor'])

hier_df = pd.DataFrame(np.random.randn(4, 5), columns=columns)
hier_df

In [ ]:
# To groupby by level, pass the level number or name using *level* keyword
hier_df.groupby(level=0, axis=1).sum()

- **Advanced GroupBy Use**
- *Group Transformation and "Unwrapped" GroupBys*
- *transform* is an instance method available to grouped objects
- It is similar to apply but imposes more constrains on the kind of function that can be used
  - It can produce a scalar value to be broadcasted to the shape if the group
  - It can produce an object of the same shape as the input group
  - It must not mutate its input

In [ ]:
df = pd.DataFrame({'key': ['a', 'b', 'c'] * 4,
                 'value': np.arange(12.)})
df.shape

In [ ]:
df

In [ ]:
g = df['value'].groupby(df['key'])
type(g)

In [ ]:
g.mean()

In [ ]:
# Suppose we want to produce a Series of the same shape as df['value'] but with values replaced by the average grouped by 'key'
g.transform(lambda x: x.mean())

- Alternatively, we can also pass built-in aggregation functions. e.g.: ```g.transform('mean')```
- - Like apply, transform works with functions that returns Series, but the result must be the *same size* as the input
- Example use cases:
  - Compute ranks in descending order for each group
  - Transform values to compute the norm or normalize each value

In [ ]:
df

In [ ]:
# Normalizing
def normalize(x):
  return (x - x.mean()) / x.std()

In [ ]:
g = df.groupby('key').value

In [ ]:
g.transform(normalize)


- Built-in aggregate functions like *mean* or *sum* are often much faster than a general apply function
- These also have a *"fast past"* when used with transform
- Allowing us to perform so-called *unwrapped* group operations:

In [ ]:
(df['value'] - g.transform('mean')) / g.transform('std')

- *Grouped Time Resampling*
- For time series data, the *resample* method is semantically a group operation based on a time intervalization

In [ ]:
N = 15
times = pd.date_range('2021-10-04 00:00', freq='1min', periods=N)
df = pd.DataFrame({'time': times,
                  'value': np.arange(N)})

# Indexing the Series using the time column
df.set_index('time', inplace=True)
print(f'shape of df: {df.shape}')
df.head()

In [ ]:
# We groupby / resample over 5 mins and see the count the number of data points
df.resample('5min').count()

- **Data Aggregation**
- Aggregation refer to any data transformation that produces scalar values from arrays
- Groupby provides multiple optimized methods for data transformation

<img src='images/group_met.png' width='450'>

- Custom aggregation functions can also be used
  - Any function that aggregates an array can be passed to the instance *aggregate* or *agg* method of group object

In [ ]:
df = pd.DataFrame({'key': ['a', 'b', 'c'] * 4,
                 'value': np.random.randint(0, 100, 12)})
df.shape

In [ ]:
df

In [ ]:
grouped = df.groupby('key')
grouped.agg(lambda x: x.max() - x.min())

In [ ]:
# Basic information of a grouped object can be retrieved using the describe instance method
grouped.describe()

- *Column-Wise and Multiple Function Application*
- A list of functions can be passed to *agg* to apply each 

In [ ]:

tips = pd.read_csv('./data/tips.csv')

# Add tip percentage of total bill
tips['tip_pct'] = tips['tip'] / tips['total_bill']

print('shape: ', tips.shape)
tips.head()

In [ ]:
# Group by day and sex
grouped = tips.groupby(['day', 'sex'], observed=False)

In [ ]:
# Checking the mean tip percentage 
grouped['tip_pct'].mean() 

In [ ]:
# Computing multiple measures over tip percentage
# when passing a lambda function, the column name is returned as lambda
#  in order to specify a column name for the result series of a lambda, 
#  we pass a tuple (name: string, lambda: function) to the aggregate method

grouped['tip_pct'].agg([('avg', 'mean'), ('maxmin', lambda x: x.max() - x.min()), 'std'])

In [ ]:
# Alternativelty a list of function can also be passed 
functions = [('avg', 'mean'), ('p2p', lambda x: x.max() - x.min()), ('sd', 'std')]
grouped[['tip_pct', 'total_bill']].agg(functions)

- **Apply General split-apply-combine**
- *apply* is the most general-purpose GroupBy method
- apply splits the object being manipulated into pieces
- invokes the passed function on each piece
- then attempts to concatenate the pieces together

In [ ]:
# A function select the rows with the largest values in a particular column
# sort the values in the column and return the last n rows
def top(df, n=5, column='tip_pct'):
  return df.sort_values(by=column)[-n:]
top(tips, n=6)

In [ ]:
# Grouping by sex and applying top
# The top function is called on each row group from the DataFrame, 
#  and then the results are glued together using pandas.concat,
#  labeling the pieces with the group names
# include_groups=False 
tips.groupby('sex').apply(top, include_groups=False) # to silence warning on adding the column names


In [ ]:
# Alternatively we can augment the default argument values for top
tips.groupby(['sex', 'day']).apply(top, n=1, column='total_bill', include_groups=False)

In [ ]:
# We could also get descriptive stats for each group using apply
tips[['sex', 'tip_pct']].groupby(['sex']).apply(lambda x: x.describe(), include_groups=False)

In [ ]:
# Disable grouping keys as index
tips.groupby(['smoker'], as_index=False).apply(top, include_groups=False)

In [ ]:
# Disable grouping keys
tips.groupby(['sex'], group_keys=False).apply(top, include_groups=False)

- *Quantile and Bucket Analysis*
- *cut* and *qcut* are tools in pandas for slicing data into buckets / bins
- These functionalities can be combined with *groupby* to perform bucket or quantile analysis on a dataset
- Categorical object returned by cut / qcut can be passed directly to groupby

In [ ]:
df = pd.DataFrame({'data1': np.random.randn(1000),
                  'data2': np.random.randn(1000)})
quartiles = pd.cut(df.data1, 4)
quartiles.value_counts()

In [ ]:
# Grouping by quartiles
grouped = df.data2.groupby(quartiles, observed=False)

In [ ]:
# What unstack does is to pivot a level of the (possibly hierarchical) index labels
# in other words, it converts a Series with a MultiIndex to a DataFrame
grouped.agg(['min', 'max', 'count', 'mean']).unstack()

- **Pivot Tables and Cross-Tabulation**
- A *pivot table* is a data summarization tool
- It aggregates a table of data by one or more keys
- It produces a result with some of the group keys along the rows and other along the columns
- Pivot tables in Python with pandas are enabled through *groupby* facility combined with *reshape* operations utilizing hierarchical indexing
- DataFrame has a *pivot_table* instance method along with the *pd.pivot_table* top-level pandas method
- pivot tables can also use partial totals called *margins*

In [ ]:
# Computing the group mean of tips DataFrame
#  The default aggregation type for the pivot table is arranged in this case by day and smoker. 
#    the aggfunc argument can be used to specify one or a list of aggregation functions. default: mean
tips[['total_bill', 'tip', 'sex', 'day', 'size', 'tip_pct']].pivot_table(index=['day', 'sex'])

In [ ]:
## Further more let us aggregate over count, sum, mean 
tips[['total_bill', 'tip', 'sex', 'day', 'size', 'tip_pct']].pivot_table(aggfunc=['count', 'sum', 'mean'], index=['day', 'sex'])


- Pivot table options

<img src='images/pivot_options.png' width=600>

- *Cross-Tabulations: Crosstab*
- A cross-tabulation (or *crosstab*) is a special case of a pivot table that computes group frequencies

In [ ]:
pd.crosstab([tips.time, tips.day, tips.smoker], tips.sex, margins=True)
